# CricVision Phase 3.4 — fine-tune YOLOv8 on the cricket dataset

Runs on a free GPU (Kaggle T4 or Colab). Reproducible: seed 42, default hyperparameters, exact pipeline in this notebook.

**Input**: `cricvision_dataset.zip` — the `data/dataset/` folder produced by `training/split_dataset.py` (YOLO layout + `dataset.yaml`), uploaded as a Kaggle Dataset or to Colab files.

**Steps**: baseline (COCO-pretrained on the test split, classes collapsed) → fine-tune `yolov8n` → fine-tune `yolov8s` → per-class test-set comparison.

**Outputs to keep**: `outputs/*_results.png` (training curves), `outputs/*_confusion_matrix.png`, `outputs/best_*.pt` (publish as a GitHub release, then set `CRICVISION_WEIGHTS`).

In [ ]:
%pip -q install ultralytics
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable the GPU accelerator')

In [ ]:
import shutil, zipfile
from pathlib import Path

SEED = 42
EPOCHS = 100
IMGSZ = 1280            # high res for the tiny ball's sake
DATASET_ZIP = Path('/kaggle/input/cricvision-dataset/cricvision_dataset.zip')  # adjust to your upload

WORK = Path('dataset')
if not WORK.exists():
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(WORK)
    while not (WORK / 'dataset.yaml').exists():  # unwrap if the zip had a top-level folder
        WORK = next(p for p in WORK.iterdir() if p.is_dir())

# rewrite the absolute path baked in by split_dataset.py to this machine
yaml_path = WORK / 'dataset.yaml'
lines = yaml_path.read_text().splitlines()
lines[0] = f'path: {WORK.resolve().as_posix()}'
yaml_path.write_text('\n'.join(lines))
print(yaml_path.read_text())

## Baseline — the number to beat

COCO-pretrained `yolov8n` on the test split with cricket classes collapsed to `person`/`sports ball` (stumps dropped: COCO can't see them). Same logic as `training/baseline_eval.py`.

In [ ]:
import sys
!git clone -q https://github.com/eklavya2201/cricvision
sys.path.insert(0, 'cricvision/training')

import tempfile
from baseline_eval import build_collapsed_split
from ultralytics import YOLO

baseline_model = YOLO('yolov8n.pt')
with tempfile.TemporaryDirectory() as tmp:
    collapsed_yaml = build_collapsed_split(WORK, Path(tmp), baseline_model.names)
    baseline = baseline_model.val(data=str(collapsed_yaml), split='test', verbose=False)
print(f'BASELINE  mAP@50={baseline.box.map50:.3f}  mAP@50-95={baseline.box.map:.3f}')

## Fine-tune

From pretrained weights, default hyperparameters first (tune only after seeing curves). Early stopping on val mAP via `patience`. Mosaic + copy-paste augmentation on for ball scarcity.

In [ ]:
runs = {}
for size in ('n', 's'):
    model = YOLO(f'yolov8{size}.pt')
    model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMGSZ, seed=SEED,
                patience=20, copy_paste=0.3, name=f'cricket_{size}', exist_ok=True)
    runs[size] = model
    print(f'yolov8{size} done')

## Evaluate on the held-out test split (per class)

In [ ]:
print(f"{'model':<10}{'class':<15}{'mAP@50':>8}{'mAP@50-95':>11}")
print(f"{'baseline':<10}{'(collapsed)':<15}{baseline.box.map50:>8.3f}{baseline.box.map:>11.3f}")
for size, model in runs.items():
    m = model.val(data=str(yaml_path), split='test', verbose=False)
    print(f"{'yolov8' + size:<10}{'ALL':<15}{m.box.map50:>8.3f}{m.box.map:>11.3f}")
    for idx, ap50, ap in zip(m.box.ap_class_index, m.box.ap50, m.box.ap):
        print(f"{'':<10}{m.names[int(idx)]:<15}{ap50:>8.3f}{ap:>11.3f}")

In [ ]:
# collect the artifacts worth keeping
out = Path('outputs')
out.mkdir(exist_ok=True)
for size in runs:
    run_dir = Path(f'runs/detect/cricket_{size}')
    shutil.copy2(run_dir / 'results.png', out / f'yolov8{size}_results.png')
    shutil.copy2(run_dir / 'confusion_matrix.png', out / f'yolov8{size}_confusion_matrix.png')
    shutil.copy2(run_dir / 'weights' / 'best.pt', out / f'best_yolov8{size}_cricket.pt')
sorted(out.iterdir())

## After the run

1. Commit `outputs/*_results.png` and `*_confusion_matrix.png` to `training/results/` in the repo.
2. Publish the winning `best_*.pt` as a GitHub release asset (and to Hugging Face Hub with a dataset card).
3. Record the baseline-vs-fine-tuned table in the README (Phase 3.5), including classes where fine-tuning *didn't* help.
4. Point the app at the new weights: `CRICVISION_WEIGHTS=path/to/best_yolov8n_cricket.pt`.